# IndicTranslate — Training Walkthrough

End-to-end: bitext → instruction rows → LoRA adapter → merged checkpoint → your own translations.

```
bitext JSONL ──render──> messages JSONL ──train──> LoRA adapter ──merge──> checkpoint
{eng, hin}                                                                     │
                                                              vllm_ready ──────┘
                                                                     │
                                                       IndicMTEngine / vllm serve
```

> **The finetuning recipe is PROVISIONAL.** It works and it is a reasonable starting point, but it
> is not a qualified configuration: no release has been trained with it from this repo, and the
> trainer behind it may be replaced. Treat every number below as a default to adjust, not as a
> setting shown to be optimal — see [docs/mt/training.md](../../docs/mt/training.md).
>
> What *is* stable is the boundary around it. The rest of the package talks to training through
> exactly two contracts: rendered `messages` JSONL in, a PEFT adapter directory out. Any trainer
> honouring those drops in without touching the engine, the prompt contract, serving or eval.

**Prerequisites**

- **GPU required** from section 4 on — sections 1–3 are pure data prep, but the setup cell asserts
  up front so you do not find out after preparing a corpus. One 80 GB card is plenty for this smoke
  run, and the shipped 8k-context recipe is sized for one.
- MT's own environment: `./install.sh && source .venv/bin/activate`
  (`transformers>=5.12`, `vllm>=0.20`, `trl==1.6.0`).
- **`bodhan-ai/indic-translate` is a public Hub repo.** For a gated one, `hf auth login` or
  `export HF_TOKEN=...` first, or point `BASE_MODEL` at a local directory. Finetuning starts from a
  model that already translates; point it at `google/gemma-4-E4B-it` only to reproduce IndicTranslate
  from scratch.

This notebook drives a **tiny smoke run** — a dozen sentence pairs and 20 optimizer steps — so
every stage finishes in minutes. The same configs scale to real corpora by editing paths, row
counts and step counts.

In [ ]:
import json
from pathlib import Path

import torch
import yaml

assert torch.cuda.is_available(), "training needs a GPU node (or the wrong venv is active)"

# --- edit these ------------------------------------------------------------
BASE_MODEL = "bodhan-ai/indic-translate"  # gated Hub repo, or a local directory
# ----------------------------------------------------------------------------

# The scripts cd to the repo root themselves, so they need an absolute path; find it
# from wherever the kernel happens to have started.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "mt").is_dir())
WORK = (REPO / "notebook_mt_train").absolute()
for d in ("bitext", "rendered", "configs", "cache"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

print("repo     :", REPO)
print("workspace:", WORK)

## 1. A bitext corpus

The pipeline starts from **bitext JSONL** — one JSON object per line with a source field and a
target field. The field *names* are yours; the render config maps them:

```json
{"eng": "The committee approved the proposal.", "hin": "समिति ने प्रस्ताव को मंजूरी दे दी।"}
```

Nothing else is required. There is no alignment step, no vocabulary build, no language tokens.

The twelve pairs below exist only to make this notebook runnable end to end. A real finetune wants
hundreds of thousands of pairs at minimum — replace `rows` with your own corpus.

In [ ]:
# (English, Hindi). The "eng" / "hin" field names chosen below are arbitrary —
# the render config maps them, so use whatever your corpus already has.
PAIRS = [
    ("The committee approved the proposal.", "समिति ने प्रस्ताव को मंजूरी दे दी।"),
    ("The meeting has been postponed.", "बैठक स्थगित कर दी गई है।"),
    ("The report has not been submitted yet.", "रिपोर्ट अभी तक जमा नहीं की गई है।"),
    ("She lives in Delhi with her family.", "वह अपने परिवार के साथ दिल्ली में रहती है।"),
    ("The train leaves at six in the morning.", "ट्रेन सुबह छह बजे रवाना होती है।"),
    ("The children are playing in the park.", "बच्चे पार्क में खेल रहे हैं।"),
    ("We will discuss this matter tomorrow.", "हम इस मामले पर कल चर्चा करेंगे।"),
    ("This school was built in 1960.", "यह स्कूल 1960 में बनाया गया था।"),
    ("The weather is very hot today.", "आज मौसम बहुत गर्म है।"),
    ("He works at a bank in the city.", "वह शहर के एक बैंक में काम करता है।"),
    ("Please close the door before you leave.", "जाने से पहले कृपया दरवाज़ा बंद कर दीजिए।"),
    ("The government announced a new scheme.", "सरकार ने एक नई योजना की घोषणा की।"),
]

rows = [{"eng": eng, "hin": hin} for eng, hin in PAIRS]
bitext = WORK / "bitext" / "eng_hin.jsonl"
bitext.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
print(f"{len(rows)} pairs -> {bitext}")

## 2. Render to instruction rows

`bodhan_genai.mt.data.render` turns bitext into the chat rows the trainer consumes. Per input row
it emits the forward direction and, with `reverse_fraction: 1.0`, the reverse one too — bitext is
normally stored one-way, and this is what makes the corpus bidirectional.

Each row draws one of **12 instruction phrasings** from `templates/variants.py` with a seeded RNG,
so the model sees paraphrase diversity rather than one memorised string. **Index 0 is exactly the
served instruction**, and the eval harness always uses index 0 — evaluation prompts sit inside the
training distribution rather than beside it.

Knobs worth knowing:

- **`template_variant: target_only`** is the released contract: names only the target language. A
  `with_source` bank exists for a single fixed-direction finetune, but a model trained that way
  must be evaluated and served the same way. Leave it alone unless you have that specific reason.
- **A dev split is required.** Checkpoint selection is by `eval_loss` and early stopping has
  nothing to watch without one. `dev_fraction` is `0.01` in the shipped config; this smoke corpus
  is 24 rows, so it is raised here to leave anything at all in dev.
- **`extra_languages`** adds a language outside the served 25 without editing the frozen contract.
- **Order matters**: do any upsampling or corpus mixing *before* this stage. The RNG advances per
  row, so N copies of a row get N different phrasings (useful augmentation); copies made afterwards
  would all share one.

In [ ]:
render_cfg = {
    "seed": 42,
    "template_variant": "target_only",  # the released contract
    "dedup": True,  # drop exact duplicate (direction, source, target) triples
    "extra_languages": {},
    "output": {
        "train": str(WORK / "rendered" / "train.jsonl"),
        "dev": str(WORK / "rendered" / "dev.jsonl"),
        "dev_fraction": 0.25,  # 0.01 in the shipped config; this corpus is 24 rows
        "dev_max_rows": 8,
    },
    "sources": [
        {
            "name": "smoke-eng-hin",
            "path": str(bitext),
            "src_field": "eng",
            "tgt_field": "hin",
            "src_lang": "eng_Latn",
            "tgt_lang": "hin_Deva",
            "reverse_fraction": 1.0,  # also emit every row hin -> eng
            "limit": None,  # cap input rows for a smoke run
        }
    ],
}
render_yaml = WORK / "configs" / "render.yaml"
render_yaml.write_text(yaml.safe_dump(render_cfg, sort_keys=False, allow_unicode=True))
print(render_yaml.read_text())

In [ ]:
# --dry-run counts the mix without writing anything. Read the stats table: a
# `*:empty` count near the row count means the field names are wrong, and you would
# otherwise get a tiny corpus and a suspiciously fast epoch.
!{REPO}/scripts/mt/render.sh {render_yaml} --dry-run

In [ ]:
!{REPO}/scripts/mt/render.sh {render_yaml}

In [ ]:
from collections import Counter

from bodhan_genai.mt import build_instruction
from bodhan_genai.mt.templates.variants import TARGET_ONLY_TEMPLATES

train_txt = (WORK / "rendered" / "train.jsonl").read_text(encoding="utf-8")
dev_txt = (WORK / "rendered" / "dev.jsonl").read_text(encoding="utf-8")
train_rows = [json.loads(line) for line in train_txt.splitlines()]
dev_rows = [json.loads(line) for line in dev_txt.splitlines()]

print(f"train {len(train_rows)} rows | dev {len(dev_rows)} rows")
print("directions:", Counter(r["direction"] for r in train_rows))
print(f"phrasings drawn: {sorted(Counter(r['template_id'] for r in train_rows))}")
print(f"of {len(TARGET_ONLY_TEMPLATES)} in the target_only bank")

row = train_rows[0]
print(f"\n--- {row['direction']}, template {row['template_id']} ---")
print("user     :", row["messages"][0]["content"])
print("assistant:", row["messages"][1]["content"])

# Only `messages` is used for training; the rest is provenance, and it is what makes a
# per-language or per-direction slice of the corpus possible afterwards.
print("\nprovenance:", {k: v for k, v in row.items() if k != "messages"})

# The contract holds: variant index 0 is byte-identical to the served instruction.
served = build_instruction("Hello world.", "hin_Deva")
variant_0 = TARGET_ONLY_TEMPLATES[0].replace("{tgt}", "Hindi").replace("{text}", "Hello world.")
print("\nindex 0 == build_instruction():", variant_0 == served)

## 3. The training config

Mirror [configs/mt/train/lora.yaml](../../configs/mt/train/lora.yaml) and shrink it. The `training:`
block is passed straight to `trl.SFTConfig` as kwargs; everything else is this repo's schema
(`bodhan_genai.mt.training.config`), which fails loudly on an unknown key.

Choices worth knowing, and what changes for a real run:

- **`target_modules: "all-linear"` is deliberate.** Gemma 4 carries `per_layer_input_gate` /
  `per_layer_projection` beside the usual q/k/v/o and gate/up/down projections, and an explicit
  list silently misses them. `exclude_modules` keeps the adapter off the vision/audio towers, which
  translation never touches.
- **`max_seq_length: 1024`** for this sentence corpus; the shipped recipe uses 8192 (documents need
  it, and `gradient_checkpointing` is on because activations dominate there). The length filter
  measures the *rendered chat*, not the raw text — set this too low against a document corpus and
  every row is filtered out, which the loader raises on rather than training on nothing.
- **`max_steps: 20`** makes this a smoke run. The shipped config uses `num_train_epochs: 3`.
- **`early_stopping_patience: 0`** disables the callback (20 steps has nothing to be patient
  about); the shipped value is 5, with `load_best_model_at_end` on `eval_loss`.
- **`report_to: "none"`** keeps wandb out of it. `scripts/mt/train_lora.sh` exports
  `WANDB_MODE=offline` by default anyway, because a training node usually has no egress and wandb
  blocking on a network call is a confusing way to discover that.
- **`max_length`, `packing` and `assistant_only_loss` are owned by the recipe.** Setting them here
  logs a warning and is ignored: packing stays off so the loss mask covers exactly one translation,
  and loss is computed on the assistant span only.
- **`eval_fraction`** expresses the eval/save cadence as a fraction of one epoch, resolved against
  the real dataset size at launch — so a config transfers between corpora instead of carrying a
  step count tuned for one. An explicit `training.eval_steps` (set below) wins.

In [ ]:
from bodhan_genai.mt.training.config import load_config

OUTPUT_DIR = WORK / "checkpoints" / "mt-lora-smoke"

train_cfg = {
    "model": {
        # Loaded with AutoModelForCausalLM — the text-only view of the multimodal checkpoint.
        "model_path": BASE_MODEL,
        "tokenizer_path": "",  # empty = use model_path
        "max_seq_length": 1024,  # 8192 in the shipped recipe
        "torch_dtype": "bfloat16",
        "attn_implementation": "sdpa",
        "gradient_checkpointing": True,
        "adapter_path": None,  # seed from an existing adapter (requires resume: false)
    },
    "lora": {
        "r": 32,
        "lora_alpha": 64,  # conventional 2 x r; effective scale = alpha/r
        "lora_dropout": 0.05,
        "bias": "none",
        "task_type": "CAUSAL_LM",
        "target_modules": "all-linear",
        "exclude_modules": [  # translation never touches the multimodal towers
            "*vision_tower*",
            "*visual*",
            "*image_encoder*",
            "*audio_tower*",
            "*embed_vision*",
            "*embed_audio*",
            "*lm_head*",
        ],
        "modules_to_save": [],
        "use_rslora": False,
        "use_dora": False,
    },
    "data": {
        "train_file": str(WORK / "rendered" / "train.jsonl"),
        "dev_file": str(WORK / "rendered" / "dev.jsonl"),
        "cache_dir": str(WORK / "cache"),  # tokenized dataset cache, file-locked
        "num_proc": 2,  # 16 in the shipped config; pointless on 24 rows
        "shuffle_seed": 42,
    },
    "resume": True,  # re-running the same output_dir continues from the last checkpoint
    "eval_fraction": 0.1,
    "training": {
        "output_dir": str(OUTPUT_DIR),
        "max_steps": 20,  # smoke run; the shipped config uses num_train_epochs: 3
        "learning_rate": 1.0e-4,  # suits LoRA; full-FT would want ~1e-5
        "lr_scheduler_type": "cosine",
        "warmup_steps": 2,
        "max_grad_norm": 5.0,  # set explicitly: frameworks disagree on the default
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 1,  # effective batch = 1 x 1 x world_size
        "per_device_eval_batch_size": 1,
        "bf16": True,
        "dataset_num_proc": 2,
        "logging_steps": 1,
        "eval_steps": 10,  # explicit; otherwise resolved from eval_fraction
        "save_steps": 10,
        "save_total_limit": 2,
        "load_best_model_at_end": False,  # true in the shipped config, on eval_loss
        "early_stopping_patience": 0,  # popped before SFTConfig; 0 = no callback
        "seed": 42,
        "report_to": "none",
    },
    "logging": {"wandb_project": "bodhan-genai-mt", "report_to": "none"},
}
# Loading it back through the real schema means a typo fails here, not 16 GB into a
# model load: the loader rejects unknown top-level and per-block keys.
train_yaml = WORK / "configs" / "train_smoke.yaml"
train_yaml.write_text(yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True))
cfg = load_config(str(train_yaml))
print("wrote", train_yaml)
print("lora r/alpha:", cfg.lora.r, "/", cfg.lora.lora_alpha, "| target:", cfg.lora.target_modules)
print("max_seq_length:", cfg.model.max_seq_length, "| resume:", cfg.resume)

## 4. Launch

`scripts/mt/train_lora.sh` wraps `accelerate launch` with `configs/mt/accelerate/single_node.yaml`
and the training environment. It checks `transformers>=5.12` up front and names the right venv if
you are in the wrong one — a pointer beats an opaque architecture error 15 GB into a model load.

The distributed strategy is **DDP, not FSDP**: only the LoRA adapter is trainable, so the optimizer
state is small and sharding would add communication for nothing. (TTS uses FSDP2 because it trains
all 3B parameters — copying that config here would be a mistake.)

`NUM_GPUS` overrides the autodetected count. **The cell blocks until training finishes** — for real
runs launch from a terminal or `tmux` instead, so the run survives a kernel restart.

Check the **trainable-parameter line** the run prints on rank 0. At r=32 with the towers excluded
it should be a low single-digit percentage; if it looks like the whole model, `peft_config` was not
applied.

In [ ]:
!NUM_GPUS=1 {REPO}/scripts/mt/train_lora.sh {train_yaml}

## 5. Checkpoints, resume, monitoring

- Checkpoints land in `training.output_dir`, with the final adapter saved to `output_dir` itself.
  A LoRA adapter is a few hundred MB, not the full model.
- **Resume**: re-run the exact same command. `resume: true` restores optimizer, scheduler, LR and
  global step from the last checkpoint in `output_dir`. Set `resume: false` to start fresh.
- **`model.adapter_path` is a different operation** — it seeds the *weights* for a fresh run with a
  new optimizer, scheduler and LR. Setting both raises, on purpose: they are easy to confuse and
  the difference is invisible afterwards.
- `bodhan_mt_run.json` is written next to the checkpoints on rank 0 and records what actually ran.
- **wandb**: `scripts/mt/train_lora.sh` exports `WANDB_MODE=offline` by default; sync later with
  `wandb sync <output_dir>/wandb/offline-run-*`. Set `logging.report_to: "none"` to disable it.

In [ ]:
ckpts = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
print("checkpoints:", [c.name for c in ckpts])
print("adapter files:", sorted(p.name for p in OUTPUT_DIR.glob("adapter*")))

# Written on rank 0 at launch: what actually ran, next to the checkpoints.
run_info = json.loads((OUTPUT_DIR / "bodhan_mt_run.json").read_text())
for key in ("model_path", "max_seq_length", "resolved_eval_steps", "train_rows", "dev_rows"):
    print(f"  {key:<20} {run_info[key]}")

ADAPTER = OUTPUT_DIR  # the final adapter; or str(ckpts[-1]) to merge a specific step

## 6. Merge the adapter into a standalone checkpoint

An adapter needs its base model at load time; vLLM wants one self-contained directory. `merge`
folds `W + (alpha/r)·B@A` back into the base weights and does three more things a naive
`merge_and_unload()` + `save_pretrained()` leaves out:

1. re-enables `use_cache` — training turns it off, and without this every generated token re-runs
   the full forward pass;
2. stages the tokenizer from the base model;
3. stages `processor_config.json` — a merged *text* model saves none, but the architecture is
   `Gemma4ForConditionalGeneration` and vLLM loads its processor.

The base model is read from the adapter's own `adapter_config.json`, not asked for, so a merge
cannot be silently pointed at the wrong base — which produces a model that loads fine and
translates badly.

`scripts/mt/merge.sh <adapter> <output>` does the merge **and** §7 in one command. It is split here
so the next step is visible, because it is the one people miss.

In [ ]:
MERGED = WORK / "merged-smoke"

!python -m bodhan_genai.mt.training.merge --adapter-path {ADAPTER} --output-dir {MERGED}

## 7. `vllm_ready` — the step that is easy to miss

**A checkpoint you trained yourself will not load on stock vLLM until this runs.** The published
checkpoint already carries the tensors; a merge does not reproduce them.

Gemma 4 E4B is a KV-sharing ("YOCO") model: `num_kv_shared_layers=18`, so its last 18 decoder
layers reuse an earlier layer's K/V and legitimately store **no `k_norm`**. vLLM builds a `k_norm`
module for *every* layer while only *using* it on non-shared layers, so its weight-load tracker
aborts:

```
ValueError: Following weights were not initialized from checkpoint:
{'model.language_model.layers.<24..41>.self_attn.k_norm.weight', ...}
```

The fix could be applied to vLLM's source, but that has to be repeated in every environment and
`pip install -U vllm` silently undoes it. This fixes the **checkpoint** instead — which travels
with the model. It writes a ~13 KB sidecar of zeroed tensors and regenerates the weight index;
the multi-GB weights and `config.json` are untouched, and re-running reports nothing to do.

Zeros are correct twice over: the values are never read on a shared layer, and Gemma's RMSNorm
computes `x * (1 + weight)`, so zero is the identity even if they were. The sizes are
heterogeneous — `full_attention` layers use `global_head_dim` (512), sliding layers `head_dim`
(256) — and getting that wrong is a load error, not a silent problem.

> If you previously *patched vLLM* to work around that error, undo it: a patched vLLM now fails on
> a converted checkpoint. `pip install --force-reinstall "vllm>=0.20"`.

In [ ]:
!python -m bodhan_genai.mt.tools.vllm_ready {MERGED} --dry-run

In [ ]:
!python -m bodhan_genai.mt.tools.vllm_ready {MERGED}

# Idempotent: re-running reports nothing to do.
print(sorted(p.name for p in MERGED.glob("*.json")))
print("sidecar present:", (MERGED / "model-shared-kv-knorm.safetensors").exists())

## 8. Translate with your own checkpoint

The merged directory is a drop-in `MODEL` argument everywhere: `IndicMTEngine`, `scripts/mt/infer.sh`,
and `CHECKPOINT=... scripts/mt/serve.sh`.

Twenty steps on twelve sentence pairs will not have taught this model anything — the point is that
the pipeline runs end to end and the artefact loads. Judge a real run with
`scripts/mt/eval.sh`, not by reading a handful of outputs.

In [ ]:
from bodhan_genai.mt import IndicMTEngine

SEGMENTS = [
    "The committee approved the proposal.",
    "Applications close at the end of the month.",
]

with IndicMTEngine(str(MERGED), max_model_len=4096) as engine:
    for r in engine.translate_batch(SEGMENTS, tgt_lang="hin_Deva"):
        print(r.text if r.ok else f"[error] {r.error}")

# Skipping the merge entirely: the HF backend loads an unmerged adapter directly,
# which is the faster loop while you are still iterating on the recipe.
#
# with IndicMTEngine(BASE_MODEL, backend="hf", adapter_dir=str(ADAPTER)) as engine:
#     print(engine.translate("The committee approved the proposal.", tgt_lang="hin_Deva").text)

## Scaling up

- **Data.** Add one entry per corpus and language pair under `sources:` in the render config;
  `dedup` is global, so the same pair appearing in two corpora is still one training example.
  Restore `dev_fraction: 0.01` with `dev_max_rows: 2000` — eval runs every few hundred steps, and a
  dev set of 1 % of a 3 M-row corpus would dominate wall-clock for no extra signal.
- **Recipe.** Restore `max_seq_length: 8192`, `num_train_epochs: 3`, `gradient_accumulation_steps: 4`,
  `lr_scheduler_type: cosine_with_min_lr` with `min_lr: 1e-5`, `warmup_steps: 0.03` (a float < 1 is
  a *fraction* of total steps in transformers 5), `load_best_model_at_end: true` on `eval_loss` and
  `early_stopping_patience: 5`. [configs/mt/train/lora.yaml](../../configs/mt/train/lora.yaml) is
  the reference. OOM at 8k: raise `gradient_accumulation_steps`, confirm `gradient_checkpointing`,
  then drop `max_seq_length` if your corpus is sentence-level.
- **Scale out.** `NUM_GPUS=8 scripts/mt/train_lora.sh <config>` for one node; multi-node swaps in
  `configs/mt/accelerate/multinode.yaml` and passes the head address at launch. Effective batch is
  `per_device × grad_accum × world_size`, so adding GPUs changes it — not a free speedup at a fixed
  learning rate. Put `data.cache_dir` somewhere every rank can see.
- **Score it, do not eyeball it.** `CHECKPOINT=<merged> scripts/mt/serve.sh`, then
  `scripts/mt/eval.sh`. The noise floor is about ±0.06 chrF++, so treat ±0.2 as a tie and prefer
  the earlier checkpoint — [docs/mt/eval.md](../../docs/mt/eval.md).
- **Inference API tour**: [notebooks/mt/inference.ipynb](inference.ipynb). Everything on one page:
  [docs/mt/end-to-end.md](../../docs/mt/end-to-end.md).